# 03 — Modeling
**Project:** Phase-Field Informed ML for Microstructure Prediction in Ti-Nb-O Refractory Alloys  
**Author:** Anosike Kelechi Kenneth  
**Purpose:** Train and evaluate Random Forest and XGBoost models on Magpie features. Compare random vs prototype-aware splitting.  
**Run order:** Run AFTER 02_eda_featurization.ipynb

---

In [ ]:
# Section 1: Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All imports OK')

In [ ]:
# Section 2: Load feature matrix saved by notebook 02
# Use relative path so the notebook runs on any machine
print('Loading Magpie feature matrix...')
df = pd.read_csv('../data/magpie_features_20k.csv')
print(f'Loaded shape: {df.shape}')

# Separate features and target
feature_cols = [c for c in df.columns if c != 'gap pbe']
X = df[feature_cols].values
y = df['gap pbe'].values

print(f'Features: {len(feature_cols)}')
print(f'Samples: {len(y)}')
print(f'Target mean: {y.mean():.3f} eV, std: {y.std():.3f} eV')

In [ ]:
# Section 3: Strategy A - Random 80/20 split (Ward et al. baseline)
# This replicates the original Ward et al. (2016) validation approach
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

# Train default Random Forest
print('Training Random Forest (n_estimators=100)...')
t0 = time.time()
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'Training time: {time.time()-t0:.1f}s')

y_pred = rf.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'\n=== Random Split Results ===')
print(f'MAE  = {mae:.4f} eV')
print(f'RMSE = {rmse:.4f} eV')
print(f'R²   = {r2:.4f}')

In [ ]:
# Section 4: Parity plot - random split
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(y_test, y_pred, c=y_test, cmap='viridis', alpha=0.4, s=10, rasterized=True)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='y = x')
plt.colorbar(sc, ax=ax, label='Actual Bandgap (eV)')
ax.set_xlabel('Actual Bandgap (eV)', fontsize=13)
ax.set_ylabel('Predicted Bandgap (eV)', fontsize=13)
ax.set_title(f'Parity Plot — Random Forest (Random Split)\nMAE={mae:.3f} eV  R²={r2:.3f}', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../figures/03_parity_random_split.png', dpi=150)
plt.show()
print('Saved: figures/03_parity_random_split.png')

In [ ]:
# Section 5: Feature importance - top 20 Magpie features
importances = rf.feature_importances_
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
ax.set_xlabel('Feature Importance (Mean Decrease Impurity)', fontsize=12)
ax.set_title('Top 20 Magpie Features — RF Bandgap Prediction', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/03_feature_importance.png', dpi=150)
plt.show()

print('Top 5 most important features:')
print(fi_df.head().to_string(index=False))

In [ ]:
# Section 6: Strategy B - Prototype-aware split (Extension Option 2 from midterm)
# Groups compounds by crystal system to prevent structural leakage
# Re-load full dataset to get crystal system labels
from matminer.datasets import load_dataset
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

print('Loading structures for crystal system extraction...')
df_full = load_dataset('matbench_mp_gap')
df_sub = df_full.sample(20000, random_state=42).reset_index(drop=True)

crystal_systems = []
for struct in df_sub['structure']:
    try:
        sga = SpacegroupAnalyzer(struct)
        crystal_systems.append(sga.get_crystal_system())
    except:
        crystal_systems.append('unknown')

groups = np.array(crystal_systems)
print('Crystal system distribution:')
unique, counts = np.unique(groups, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

In [ ]:
# Section 7: Train RF on prototype-aware split and compare
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf_p = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_p.fit(X[train_idx], y[train_idx])
y_pred_p = rf_p.predict(X[test_idx])

mae_p  = mean_absolute_error(y[test_idx], y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y[test_idx], y_pred_p))
r2_p   = r2_score(y[test_idx], y_pred_p)
pct    = (mae_p - mae) / mae * 100

print(f'\n=== Strategy Comparison ===')
print(f'{"Metric":<8} {"Random Split":<18} {"Prototype-Aware":<18} {"% Change"}')
print(f'{"MAE":<8} {mae:<18.4f} {mae_p:<18.4f} {pct:+.1f}%')
print(f'{"RMSE":<8} {rmse:<18.4f} {rmse_p:<18.4f}')
print(f'{"R²":<8} {r2:<18.4f} {r2_p:<18.4f}')

In [ ]:
# Section 8: Side-by-side parity plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, yt, yp, title in [
    (axes[0], y_test, y_pred, f'Random Split\nMAE={mae:.3f} eV  R²={r2:.3f}'),
    (axes[1], y[test_idx], y_pred_p, f'Prototype-Aware Split\nMAE={mae_p:.3f} eV  R²={r2_p:.3f}')
]:
    sc = ax.scatter(yt, yp, c=yt, cmap='viridis', alpha=0.35, s=8, rasterized=True)
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='y = x')
    ax.set_xlabel('Actual Bandgap (eV)', fontsize=12)
    ax.set_ylabel('Predicted Bandgap (eV)', fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend()
    plt.colorbar(sc, ax=ax, label='Actual Bandgap (eV)')

plt.suptitle('Random Forest: Random vs Prototype-Aware Split', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/03_parity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/03_parity_comparison.png')
print('\nNext step: Run 04_results_visualization.ipynb')